# Partycja per device i day oraz wymuszony kierunek DESC clustering po event_time
Wszystkie dane urządzenia trafiają teraz na partycję złożoną (device_id, day), co sprawia, że odczyt staje się szybszy.
Dane posortowane są w kolejności od najnowszych do najstarszych.
Partition key = device_id, day
Clustering key = event_time DESC

In [ ]:
import uuid
from cassandra.cluster import Cluster

In [ ]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

In [ ]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

In [ ]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_4 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY ((device_id, day), event_time)
    ) WITH CLUSTERING ORDER BY (event_time DESC)
""")
print("Utworzono tabelę 'users'")

## Generate test data

In [ ]:
!python /workspace/_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_4 \
  --records 1000 \
  --batch-size 100

### Podstawienie pełnego klucza

In [ ]:
rows = session.execute("""
SELECT *
FROM
    events_by_device_4
WHERE
    device_id='device_1'
    AND day='2026-03-29'
LIMIT 10
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

### Podstawienie tylko device_id

In [ ]:
rows = session.execute("""
SELECT *
FROM
    events_by_device_4
WHERE
    device_id='device_1'
LIMIT 10
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

### Brak podstawienie partition key

In [ ]:
try:
    session.execute("""
    SELECT *
    FROM
        events_by_device_4
    WHERE
        temperature > 25
    """)
except Exception as e:
    print(e)

### ALLOW FILTERING

In [ ]:
rows = session.execute("""
SELECT *
FROM
    events_by_device_4
WHERE
    temperature > 25 ALLOW FILTERING
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

In [ ]:
# Zamknięcie połączenia
cluster.shutdown()